# 03 — ENSO-phase composites of moisture sources

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/03_enso_composites.ipynb)

**What we're investigating.** Seasonal precipitation in much of South
America is modulated by ENSO (El Niño / La Niña). This notebook computes
**seasonal moisture-source composites** for Central Chile under El Niño
vs La Niña years, asking whether the upwind moisture sources (and the
share of land-vs-ocean evaporation feeding the rain) shift between
phases.

**The visualisation choice.** We plot per-panel **spatial rank-CDFs**
rather than absolute mm/month. This means each panel is normalised to
its own daily total: 0% = the dominant source pixel, 100% = the long
tail. The advantage is that small absolute differences between phases
show up as clear contour-level shifts even when one phase has much more
rain in absolute terms than the other.

**Subset for Colab.** A full reproduction would cover 30 years x 4
seasons x 3 phases. To keep this notebook responsive on Colab we cap
to **2 phases x 4 seasons x 2 years each**:

- El Niño:  1997, 1998  (strong East-Pacific event)
- La Niña:  2010, 2011  (sustained La Niña)

**Dataset.** AguaTrack v1, **monthly aggregate** zarr stores
(`{year}_monthly_sum.zarr`). Same schema as the daily store but with
`time` collapsed to 12 monthly sums per year.

**How to cite.** TODO: paper in submission.

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrack/AguaTrack-ARCO-SA`. Monthly
  aggregate sub-directory is not yet uploaded.
- **Receptor box** — Central Chile (Santiago region) by default. Set
  `LAT_MIN`/`LAT_MAX`/`LON_MIN`/`LON_MAX` to any other South American
  sink to compare its ENSO sensitivity.
- **ENSO phase year sets** — capped to 2 years per phase for Colab.
  Adding more years gives a smoother composite at the cost of longer
  read time.

In [ ]:
HF_REVISION = "main"

LAT_MIN, LAT_MAX = -35.0, -32.0
LON_MIN, LON_MAX = -72.0, -70.0
study_area = "Central Chile"

# ENSO phase years (capped subset for Colab).
EL_NINO_YEARS: set[int] = {1997, 1998}
LA_NINA_YEARS: set[int] = {2010, 2011}
ALL_YEARS = sorted(EL_NINO_YEARS | LA_NINA_YEARS)

# Convenience: phase -> sorted year list, plus the iteration order used
# everywhere below. Defining it here keeps every cell that follows
# independent of cell order.
phase_sets = {
    "El Niño": sorted(EL_NINO_YEARS),
    "La Niña": sorted(LA_NINA_YEARS),
}
phase_names = list(phase_sets.keys())

# Full zarr URLs, one per year. The monthly-aggregate sub-directory is
# not yet on HuggingFace — these URLs will start resolving once it is
# uploaded under the same naming scheme.
AGUATRACK_MONTHLY_URLS = [
    f"hf://datasets/AguaTrack/AguaTrack-ARCO-SA/AguaTrack_ARCO_SA_monthly/{yr}_monthly_sum.zarr"
    for yr in ALL_YEARS
]

# Seasons follow the standard meteorological convention.
SEASONS = ["DJF", "MAM", "JJA", "SON"]
# Month indices into a 12-element annual time axis (Jan=0, ..., Dec=11).
SEASON_MONTHS = {
    "DJF": [0, 1, 11],
    "MAM": [2, 3, 4],
    "JJA": [5, 6, 7],
    "SON": [8, 9, 10],
}

# Colour scheme used in the bar chart at the end.
PHASE_COLORS = {"El Niño": "#d73027", "La Niña": "#4575b4"}


def classify_year(yr: int) -> str:
    """Map a year to its ENSO phase label."""
    if yr in EL_NINO_YEARS:
        return "El Niño"
    if yr in LA_NINA_YEARS:
        return "La Niña"
    return "Neutral"

## Step 2 — Install dependencies (Colab only)

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

## Step 3 — Imports and plotting style

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr
from matplotlib.patches import Rectangle

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

## Step 4 — Find tag cells inside the receptor box

We use the 1997 store as the coordinate reference (`tag_lat`/`tag_lon`
are stable across years). Boolean indexing gives us a mask we can
pass directly to `isel`.

In [ ]:
ds_ref = xr.open_zarr(AGUATRACK_MONTHLY_URLS[0],
                      storage_options={"revision": HF_REVISION})
in_box = (
    (ds_ref.tag_lat >= LAT_MIN) & (ds_ref.tag_lat <= LAT_MAX)
    & (ds_ref.tag_lon >= LON_MIN) & (ds_ref.tag_lon <= LON_MAX)
)
box_tag_idx = np.flatnonzero(in_box.values)
print(f"tag cells in box: {len(box_tag_idx)}")

## Step 5 — Build the seasonal lazy graph and materialise once

Pangeo idiom: stack the 4 yearly stores into one virtual dataset along
a new `year` dim, then sum over the box's tags lazily (this collapses
all of central Chile into a single moisture sink). We also build the
continental-recycling ratio (e_track over land / total e_track) lazily
for the bar chart at the end.

A single `.load()` with a progress bar materialises everything in one
pass — dask schedules the 4 yearly reads concurrently.

In [ ]:
from dask.diagnostics import ProgressBar

# Stack lazy datasets along a new "year" dim. Pre-select the box tags
# before any read.
ds_all = xr.concat(
    [
        xr.open_zarr(url, storage_options={"revision": HF_REVISION})
            .isel(tagging_mask=box_tag_idx)
        for url in AGUATRACK_MONTHLY_URLS
    ],
    dim=xr.Variable("year", ALL_YEARS),
)
lsm = ds_ref.lsm

# Lazy: monthly source map and monthly tagged-precip, summed over tags.
src_monthly = ds_all.e_track.sum(dim="tagging_mask")           # (year, time, lat, lon)
precip_monthly = ds_all.tagged_precip.sum(dim="tagging_mask")  # (year, time)

# Lazy: per-(year, season) composite via a dict-comprehension over months.
src_seasonal_lazy = xr.concat(
    [src_monthly.isel(time=SEASON_MONTHS[s]).mean(dim="time") for s in SEASONS],
    dim=xr.Variable("season", SEASONS),
)
precip_seasonal_lazy = xr.concat(
    [precip_monthly.isel(time=SEASON_MONTHS[s]).mean(dim="time") for s in SEASONS],
    dim=xr.Variable("season", SEASONS),
)

# Lazy: annual continental-recycling ratio.
total_annual = src_monthly.sum(dim=("time", "latitude", "longitude"))
land_annual = src_monthly.where(lsm).sum(dim=("time", "latitude", "longitude"))
ratio_lazy = land_annual / total_annual.where(total_annual > 0)

# Materialise everything at once.
with ProgressBar():
    src_seasonal = src_seasonal_lazy.load()      # (season, year, lat, lon)
    precip_seasonal = precip_seasonal_lazy.load()  # (season, year)
    year_ratios = ratio_lazy.load()              # (year,)

ds_ref.close()
print(f"loaded {src_seasonal.sizes['year']} years x {src_seasonal.sizes['season']} seasons "
      f"({src_seasonal.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Spatial CDF helper

Each panel is ranked independently. CDF(pixel) = "this pixel and all
stronger-contributing pixels together account for X% of the panel
total." 0% = a single dominant source pixel; 100% = the long tail.
When you draw a contour at, say, 30%, you get the boundary of "the
pixels that supply 30% of the source".

In [ ]:
def spatial_cdf(da: xr.DataArray) -> xr.DataArray:
    """Convert a 2-D positive-valued source map into a per-day rank CDF in %."""
    vals_flat = da.values.flatten()
    valid_mask = ~np.isnan(vals_flat) & (vals_flat > 0)
    valid_vals = vals_flat[valid_mask]
    # Rank descending and accumulate.
    sort_idx = np.argsort(valid_vals)[::-1]
    cumsum_pct = np.cumsum(valid_vals[sort_idx]) / np.sum(valid_vals[sort_idx]) * 100
    # Scatter the cumulative percentages back to their pixel positions.
    cdf_flat = np.full_like(vals_flat, np.nan, dtype=float)
    cdf_flat[np.where(valid_mask)[0][sort_idx]] = cumsum_pct
    return xr.DataArray(cdf_flat.reshape(da.values.shape), coords=da.coords, dims=da.dims)

## Step 7 — The 2 x 4 ENSO-phase x season composite grid

Rows = ENSO phases, columns = seasons. Each panel shows the source CDF
for that phase x season composite, with the receptor box outlined in
red. The label box in the top-left corner of each row gives the phase
name and the years included; the small dark box in the bottom-right of
every panel reports the composite mean tagged precipitation in mm/mo.

In [ ]:
# Pre-compute (phase, season) composites in pure xarray. For each phase
# we slice `src_seasonal` / `precip_seasonal` on the `year` dim and
# average along it.
mean_maps = {
    (phase, season): src_seasonal.sel(year=yrs, season=season).mean(dim="year")
    for phase, yrs in phase_sets.items()
    for season in SEASONS
}
precip_composite = {
    (phase, season): float(precip_seasonal.sel(year=yrs, season=season).mean())
    for phase, yrs in phase_sets.items()
    for season in SEASONS
}

# Convert each composite to its per-panel rank CDF.
cdf_grid = {k: spatial_cdf(v) for k, v in mean_maps.items()}
levels = np.linspace(0, 100, 11)


def style_ax(ax, left_labels=True, bottom_labels=True):
    """Decorate one panel of the grid (coastlines, extent, gridlines)."""
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    ax.set_extent([-90, -30, -55, 15], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.1)
    gl.top_labels = gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))


fig, axes = plt.subplots(
    len(phase_names), len(SEASONS),
    figsize=(16, 8),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = np.atleast_2d(axes)  # so the (rows, cols) indexing works for 1-row layouts too

cf = None
for r, phase in enumerate(phase_names):
    for c, season in enumerate(SEASONS):
        ax = axes[r, c]
        # Filled contours of the rank CDF for this panel.
        cf = cdf_grid[(phase, season)].plot.contourf(
            ax=ax, levels=levels, cmap=cmc.batlowW_r,
            transform=ccrs.PlateCarree(), add_colorbar=False,
        )
        # Outline the receptor box.
        ax.add_patch(Rectangle(
            (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
            linewidth=2, edgecolor="red", facecolor="none",
            transform=ccrs.PlateCarree(), zorder=5,
        ))
        # Lat labels only on the leftmost column; lon labels only on the bottom row.
        style_ax(ax, left_labels=(c == 0), bottom_labels=(r == len(phase_names) - 1))
        # Season name as the column header (top row only).
        ax.set_title(season if r == 0 else "", fontweight="bold")
        # Phase + years label on the leftmost column only.
        if c == 0:
            ax.text(
                -88, 10,
                f"{phase} [{', '.join(str(y) for y in sorted(phase_sets[phase]))}]",
                transform=ccrs.PlateCarree(), zorder=10,
                fontweight="bold", color="white", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="#333333", alpha=0.75),
            )
        # Mean precipitation chip in the bottom-right of every panel.
        ax.text(
            -32, -53,
            f"{int(precip_composite[(phase, season)])} mm/mo",
            transform=ccrs.PlateCarree(), zorder=10,
            color="white", ha="right", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="#1a1a1a", alpha=0.70),
        )

# One shared horizontal colour bar at the bottom.
fig.subplots_adjust(left=0.06, right=0.98, top=0.92, bottom=0.08, hspace=0.06, wspace=0.04)
cbar_ax = fig.add_axes([0.15, 0.02, 0.7, 0.012])
fig.colorbar(cf, cax=cbar_ax, orientation="horizontal",
             label="Cumulative Moisture Contribution (%)")
fig.suptitle(
    f"ENSO-Phase Seasonal Precipitationsheds — Spatial CDF | {study_area}\n"
    "(each panel ranked independently within its own phase x season composite)",
    y=1.0,
)

OUT = Path("outputs/enso"); OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig1_enso_composites.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 8 — Annual continental recycling ratio

The continental-recycling ratio is the fraction of tagged moisture
sourced from land (vs ocean). Higher = more locally recycled. We
bar-chart the four selected years and colour the bars by ENSO phase;
dashed lines mark the per-phase mean.

In [ ]:
# year_ratios is a DataArray indexed by `year`. Convert to % for plotting.
ratios_pct = year_ratios * 100
years = ratios_pct.year.values
colors = [PHASE_COLORS[classify_year(int(y))] for y in years]

# Per-phase mean recycling ratios via xarray slicing.
phase_means = {
    p: float(ratios_pct.sel(year=phase_sets[p]).mean())
    for p in phase_names
}

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(years, ratios_pct.values, color=colors, alpha=0.85)
for phase, mean_val in phase_means.items():
    ax.axhline(mean_val, color=PHASE_COLORS[phase], linestyle="--", lw=1.5)
patches = [
    mpatches.Patch(color=PHASE_COLORS[p], label=f"{p}  (mean {phase_means[p]:.1f}%)")
    for p in phase_names
]
ax.legend(handles=patches, loc="upper right")
ax.set_xlabel("Year")
ax.set_ylabel("Continental Recycling Ratio (%)")
ax.set_title(f"ENSO Influence on Continental Moisture Recycling\n{study_area}")
fig.savefig(OUT / "fig2_recycling_ratio_timeseries.png", bbox_inches="tight", dpi=150)
plt.show()

# Quick summary statistic: how different are the two phases?
anomaly = phase_means["El Niño"] - phase_means["La Niña"]
print(f"\nEl Niño - La Niña recycling-ratio anomaly: {anomaly:+.2f} percentage points")